# Curve Preprocessing Walkthrough (no NC subtraction)

Picks one real pixel from a POC_DDM_final experiment and tracks it through 01_curve_preprocessing_v6.py's real pipeline, using the exact same functions the pipeline uses (imported directly -- nothing reimplemented). X-axis is real experiment time throughout.

**This variant skips NC subtraction** (matches `01_curve_preprocessing_v6.py` run without `--nc_subtract`) -- see `1_curve_preprocessing_walkthrough.ipynb` for the NC-subtracted version.

**Stages shown:**
1. Raw signal (pre-linearisation) -- captured via a hook into linearise.linearise(), since the pipeline never exposes this intermediate directly.
2. Linearised signal -- marked with where 01's truncation index (idx_settled + max_significant_index) and idx_end fall on this un-windowed timeline.
3. Truncated + Rebaselined -- cropped to 01's max_significant_index, then rebaselined (curve starts at 0); NC subtraction is skipped in this notebook.
4. Denoising -- Savitzky-Golay filter, computed on the full batch (so its auto window-sweep sees a realistic sample size) and read back for our one pixel.

**To try a different sample:** change EXP_NAME / WELL_ID / PIXEL_SEED in the "Pick a sample" cell and re-run from there.


In [ ]:
import os, sys, importlib.util, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Run from main/ (matches house style -- 01_curve_preprocessing_v6.py's own
# sys.path.insert calls are CWD-relative, not __file__-relative).
try:
    _nb = globals().get('__vsc_ipynb_file__')
    if _nb:
        os.chdir(os.path.dirname(os.path.dirname(os.path.dirname(_nb))))
except Exception:
    pass
sys.path.insert(0, os.getcwd())
sys.path.insert(0, 'utils')
sys.path.insert(0, 'utils/model_training')

%load_ext autoreload
%autoreload 2

_spec = importlib.util.spec_from_file_location(
    "chip_preprocessing", str(Path.cwd() / "chip" / "preprocessing.py"))
cp01 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cp01)
config = cp01.config

# cp01's own top-level imports (chip_v6_utils -> titan_v6/__init__ path insert) put
# titan_v6 on sys.path as a side effect, so this now resolves.
import linearise as linearise_mod

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linestyle': '--',
    'grid.alpha': 0.7,
    'font.size': 10,
})
print("CWD:", os.getcwd())
print("Available POC_DDM_final experiments:",
      sorted(config.CROSS_DATASET_GROUPS['final_6_new']))


## Pick a sample -- change these and re-run from here

In [ ]:
EXP_NAME = 'D20260827_E00_C00_F4500KHz_U_DDM_01_final_final'  # any name from the list printed above
exp_path = Path(config.FINAL_EXP_FOLDER) / EXP_NAME


## Load + capture the raw (pre-linearisation) signal

`linearise.linearise(frame_3d, params, idx_active_gain)` is where the raw ADC-ish signal
becomes the linearised chemical signal (see `titan_v6/linearise.py`). The pipeline only
ever keeps the *output* on the `Well` object, so we hook the function to also record its
*input* for the one call this experiment makes.

In [ ]:
_captured = []
_orig_linearise = linearise_mod.linearise
def _hooked_linearise(frame_3d, *args, **kwargs):
    raw_in = frame_3d.copy()
    out = _orig_linearise(frame_3d, *args, **kwargs)
    lin_out = out[0].copy() if isinstance(out, tuple) else out.copy()
    _captured.append((raw_in, lin_out))
    return out
linearise_mod.linearise = _hooked_linearise

t0 = time.perf_counter()
all_exp_data = cp01.load_and_preprocess_v6(
    exp_path, n_wells=config.N_WELLS, n_a_type=config.N_A_TYPE, vref_ref_idx='all')
linearise_mod.linearise = _orig_linearise  # unhook, one-shot capture is enough
print(f"\nLoaded in {time.perf_counter() - t0:.1f}s -- {len(all_exp_data)} vref slice(s), "
      f"{len(all_exp_data[0].wells_list)} wells")

raw_full, lin_full = _captured[0]  # full-chip (rows, cols, time), before the well split


## Locate our pixel in the full-chip array

`preprocess.split_wells` splits the full chip into a `(n_wells//2, 2)` grid of equal
row/col blocks (verified directly: `well.well_3d_lin[local_r, local_c, :]` exactly
matches `lin_full[global_r, global_c, :]` under this mapping). This lets us track one
*real* physical pixel continuously from the raw signal all the way through.

In [ ]:
WELL_ID = 0           # 0..9 (config.N_WELLS=10 wells per experiment)
well = all_exp_data[0].wells_list[WELL_ID]
local_rows, local_cols = well.well_3d_lin.shape[0], well.well_3d_lin.shape[1]

idx_active = np.asarray(well.idx_active)
active_positions = np.where(idx_active)[0]


In [ ]:
PIXEL_SEED = 1        # picks a random *active* pixel within that well; change for a different curve
rng = np.random.default_rng(PIXEL_SEED)
pick_flat = int(rng.choice(active_positions))
local_r, local_c = np.unravel_index(pick_flat, (local_rows, local_cols), order='C')

row_group, col_group = WELL_ID // 2, WELL_ID % 2
global_r = row_group * local_rows + local_r
global_c = col_group * local_cols + local_c
print(f"well={WELL_ID}  local pixel=({local_r},{local_c})  global pixel=({global_r},{global_c})")


## Reconstruct the full batch + find 01's truncation index (MSI)

Both need the *whole* experiment's active-pixel curves, not just our one pixel -- the
same array `01_curve_preprocessing_v6.py`'s main block works with, so the truncation
index is computed exactly the way the real pipeline computes it.

In [ ]:
X_time, Y_well, X_2d_bs_active = cp01.reconstruct_data(all_exp_data, attr_str="well_2d_bs_active")

# Row index of our pixel within the reconstructed batch: reconstruct_data iterates
# wells in order, each contributing its active pixels in idx_active's True-position
# order -- so it's (# active pixels in earlier wells) + (# active pixels before ours
# within this well).
active_rank = int(np.sum(idx_active[:pick_flat]))
offset = sum(int(np.sum(all_exp_data[0].wells_list[i].idx_active)) for i in range(WELL_ID))
row_idx = offset + active_rank
assert np.allclose(X_2d_bs_active[row_idx], well.well_2d_bs_active[:, np.sum(idx_active[:pick_flat])]), \
    "pixel row mapping is wrong -- fix before trusting downstream steps"
print(f"X_2d_bs_active shape: {X_2d_bs_active.shape}  (our pixel is row {row_idx})")


In [ ]:
from scipy.ndimage import uniform_filter1d

mapping = config.LABEL_MAPPINGS.get(EXP_NAME)
y_label = np.array([mapping.get(w, w) for w in Y_well]) if mapping is not None else None
pc_mask = (y_label == 'PC') if y_label is not None else np.zeros(len(Y_well), dtype=bool)

if np.any(pc_mask):
    smoothed = uniform_filter1d(np.mean(X_2d_bs_active[pc_mask], axis=0).squeeze(), size=20)
    max_significant_index = int(np.argmin(smoothed))
    print(f"[PC argmin] max_significant_index={max_significant_index} ({int(pc_mask.sum())} PC-well curves)")
else:
    ori_curve_dydx = np.array(cp01.get_derivatives(X_2d_bs_active, X_time))
    min_indices = np.argmin(ori_curve_dydx, axis=1)
    unique_indices, counts = np.unique(min_indices, return_counts=True)
    significant_indices = unique_indices[counts > (len(ori_curve_dydx) / 3)]
    max_significant_index = int(np.max(significant_indices)) if len(significant_indices) > 0 else None
    print(f"[Derivative vote] max_significant_index={max_significant_index}")


## 1-2. Raw and linearised

X-axis is real experiment time throughout (`well.time_npr`, seconds) so every panel is
directly comparable. The linearised panel marks where `01`'s truncation index sits on
this un-windowed timeline (`idx_settled + max_significant_index`) and where `idx_end`
is -- both computed on the *full* raw/linearised signal, not the truncated one, since
that's the only timeline this panel can show.

In [ ]:
t_full = well.time_npr  # full, un-windowed experiment time axis (seconds)

steps = []  # list of (title, x, y) -- built up one stage at a time
steps.append(("1. Raw Signal", t_full, raw_full[global_r, global_c, :]))
steps.append(("2. Linearised", t_full, lin_full[global_r, global_c, :]))

msi_time = t_full[well.idx_settled + max_significant_index]
idx_end_time = t_full[min(well.idx_end, len(t_full) - 1)]
print(f"MSI marker at t={msi_time:.1f}s (idx_settled={well.idx_settled} + MSI={max_significant_index})")
print(f"idx_end marker at t={idx_end_time:.1f}s (idx_end={well.idx_end})")


## 3. Truncated + Rebaselined

Cropped to `01`'s `max_significant_index`, then rebaselined so each curve starts at 0 --
matching the real pipeline's order (rebaseline happens right after truncation). NC
subtraction is skipped in this notebook.


In [ ]:
X_2d_trunc, t_trunc = X_2d_bs_active, t_full[well.idx_settled:well.idx_end]
if max_significant_index is not None and (max_significant_index + 1) < (len(X_time) - 100):
    X_2d_trunc = X_2d_bs_active[:, max_significant_index + 1:]
    t_trunc = t_trunc[max_significant_index + 1:]
t_trunc = t_trunc - t_trunc[0]  # matches 01's own post-truncation timestamp reset

X_2d_trunc = X_2d_trunc - X_2d_trunc[:, 0:1]  # rebaseline -- real pipeline does this right after truncation

# NC subtraction skipped in this notebook (matches 01_curve_preprocessing_v6.py without --nc_subtract).
X_2d_nc = X_2d_trunc.copy()

steps.append(("3. Truncated + Rebaselined", t_trunc, X_2d_nc[row_idx]))
print("Truncated shape:", X_2d_trunc.shape)


## 4. Denoising -- Savitzky-Golay Filter


In [ ]:
t0 = time.perf_counter()
sg_curves, opt_w = cp01.sg_p4_denoise_curves(X_2d_nc)
steps.append((f"4. Denoising - Savitzky-Golay Filter", t_trunc, sg_curves[row_idx]))
print(f"SG denoise done in {time.perf_counter() - t0:.1f}s")


## Plot every stage

In [ ]:
from matplotlib.ticker import MultipleLocator

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    # 'axes.grid': True,
    # 'grid.color': '#e0e0e0',
    # 'grid.linestyle': '--',
    # 'grid.alpha': 0.7,
    'font.size': 10,
})

print(f"{EXP_NAME}  |  well {WELL_ID}  |  pixel (local {local_r},{local_c} / "
      f"global {global_r},{global_c})")

colors = plt.cm.viridis(np.linspace(0.3, 0.5, len(steps)))

for i, ((title, x, y), color) in enumerate(zip(steps, colors)):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.plot(x, y, color=color, lw=2)
    # ax.set_title(title, fontsize=10.5, fontweight='bold', loc='left')
    ax.set_xlim(x[0], x[-1])
    ax.set_xlabel("Time (s)", fontsize=11)
    ax.tick_params(labelsize=11)
    ax.tick_params(axis='x', rotation=60)
    ax.xaxis.set_major_locator(MultipleLocator(300))
    if i == 1:  # linearised panel -- mark where MSI truncation and idx_end fall
        ylim = ax.get_ylim()
        ax.axvline(msi_time, color='#c0392b', lw=2, linestyle='--', zorder=5)
        ax.axvline(idx_end_time, color='#c0392b', lw=2, linestyle='--', zorder=5)
        ax.text(msi_time, ylim[1], ' settled', color='#c0392b', fontsize=11, va='top', ha='left')
        ax.text(idx_end_time, ylim[1], 'end ', color='#c0392b', fontsize=11, va='top', ha='right')
    fig.tight_layout()
    plt.show()


## Combined pipeline figure (report)

Single-row, 4-panel version of the plot above, formatted for the report: bold
captions below each panel and arrows between stages instead of per-panel titles.
Saved as a high-res PNG (300 dpi) to `notebooks/figures/`.


In [ ]:
panel_titles = ['Raw Curve', 'Linearised Curve', 'Truncated Curve', 'Denoised Curve']
colors = plt.cm.viridis(np.linspace(0.3, 0.5, len(steps)))

fig, axes = plt.subplots(1, len(steps), figsize=(4.5 * len(steps), 4.3), dpi=300)
fig.subplots_adjust(wspace=0.35, bottom=0.22, top=0.95)

for i, ((title, x, y), color, ax) in enumerate(zip(steps, colors, axes)):
    ax.plot(x, y, color=color, lw=2)
    ax.set_xlim(x[0], x[-1])
    ax.tick_params(labelsize=10)
    ax.tick_params(axis='x', rotation=60)
    ax.xaxis.set_major_locator(MultipleLocator(300))
    if i == 1:  # linearised panel -- mark where MSI truncation and idx_end fall
        ylim = ax.get_ylim()
        ax.axvline(msi_time, color='#c0392b', lw=2, linestyle='--', zorder=5)
        ax.axvline(idx_end_time, color='#c0392b', lw=2, linestyle='--', zorder=5)
        ax.text(msi_time, ylim[1], ' settled', color='#c0392b', fontsize=10, va='top', ha='left')
        ax.text(idx_end_time, ylim[1], 'end ', color='#c0392b', fontsize=10, va='top', ha='right')

fig.canvas.draw()  # finalize subplot positions before reading them for captions

# Caption below each panel, anchored to its *tight* bbox (spine + rotated tick
# labels) rather than a fixed y -- keeps it clear of the x-axis regardless of
# tick rotation/fontsize instead of needing a hand-tuned constant.
renderer = fig.canvas.get_renderer()
for ax, label in zip(axes, panel_titles):
    pos = ax.get_position()
    tb  = ax.get_tightbbox(renderer).transformed(fig.transFigure.inverted())
    fig.text((pos.x0 + pos.x1) / 2, tb.y0 - 0.02, label, ha='center', va='top', fontsize=16)

SAVE_DIR = Path('figures')
SAVE_DIR.mkdir(exist_ok=True)
# save_path = SAVE_DIR / f'{EXP_NAME}_pipeline_stages.png'
# fig.savefig(save_path, dpi=300, facecolor='white')
# print(f"Saved high-res figure -> {save_path.resolve()}")
plt.show()


## Multi-chip pipeline comparison (LOFO group)

Extends the single-chip walkthrough above to every chip in a LOFO group, for one
selected target's real pixel curve per chip -- through raw, linearised, truncated,
denoised, **normalised** (per-curve min-max), and **temporal-calibrated** (PC-TTP
`anchor="min"` alignment, same logic as `04_cross_dataset_training.py`'s
`shift_to_pc_ttp_anchor` / `truncate_to_common_duration`). One 2x3 figure, one
consistent colour per chip across all six panels.

**To try a different group/target:** change `LOFO_GROUP` / `TARGET_LABEL` in the cell below.

In [ ]:
from scipy.ndimage import uniform_filter1d
from chip import cross_dataset_training as cdt

LOFO_GROUP = 'final_6_new'    # any key in config.CROSS_DATASET_GROUPS
TARGET_LABEL = 'IAV'          # any label present in these chips' LABEL_MAPPINGS
chip_names = config.CROSS_DATASET_GROUPS[LOFO_GROUP]
print(f"LOFO_GROUP={LOFO_GROUP!r}  TARGET_LABEL={TARGET_LABEL!r}  ({len(chip_names)} chips)")
for name in chip_names:
    print(' ', name)

In [ ]:
def run_pixel_pipeline(exp_name, target_label, pixel_seed=1):
    exp_path = Path(config.FINAL_EXP_FOLDER) / exp_name

    _captured = []
    _orig_linearise = linearise_mod.linearise
    def _hooked_linearise(frame_3d, *args, **kwargs):
        raw_in = frame_3d.copy()
        out = _orig_linearise(frame_3d, *args, **kwargs)
        lin_out = out[0].copy() if isinstance(out, tuple) else out.copy()
        _captured.append((raw_in, lin_out))
        return out
    linearise_mod.linearise = _hooked_linearise
    all_exp_data = cp01.load_and_preprocess_v6(
        exp_path, n_wells=config.N_WELLS, n_a_type=config.N_A_TYPE, vref_ref_idx='all')
    linearise_mod.linearise = _orig_linearise
    raw_full, lin_full = _captured[0]

    X_time, Y_well, X_2d_bs_active = cp01.reconstruct_data(all_exp_data, attr_str="well_2d_bs_active")

    mapping = config.LABEL_MAPPINGS.get(exp_name)
    y_label = np.array([mapping.get(w, w) for w in Y_well]) if mapping is not None else Y_well
    matches = np.where(y_label == target_label)[0]
    if len(matches) == 0:
        raise ValueError(f"{exp_name}: no active pixel labelled {target_label!r} "
                         f"(available: {sorted(set(y_label))})")
    WELL_ID = int(Y_well[matches[0]])

    well = all_exp_data[0].wells_list[WELL_ID]
    local_rows, local_cols = well.well_3d_lin.shape[0], well.well_3d_lin.shape[1]
    idx_active = np.asarray(well.idx_active)
    active_positions = np.where(idx_active)[0]

    rng = np.random.default_rng(pixel_seed)
    pick_flat = int(rng.choice(active_positions))
    local_r, local_c = np.unravel_index(pick_flat, (local_rows, local_cols), order='C')
    row_group, col_group = WELL_ID // 2, WELL_ID % 2
    global_r = row_group * local_rows + local_r
    global_c = col_group * local_cols + local_c

    active_rank = int(np.sum(idx_active[:pick_flat]))
    offset = sum(int(np.sum(all_exp_data[0].wells_list[i].idx_active)) for i in range(WELL_ID))
    row_idx = offset + active_rank
    assert np.allclose(X_2d_bs_active[row_idx], well.well_2d_bs_active[:, np.sum(idx_active[:pick_flat])]), \
        "pixel row mapping is wrong -- fix before trusting downstream steps"

    pc_mask = (y_label == 'PC') if mapping is not None else np.zeros(len(Y_well), dtype=bool)
    if np.any(pc_mask):
        smoothed = uniform_filter1d(np.mean(X_2d_bs_active[pc_mask], axis=0).squeeze(), size=20)
        max_significant_index = int(np.argmin(smoothed))
    else:
        ori_curve_dydx = np.array(cp01.get_derivatives(X_2d_bs_active, X_time))
        min_indices = np.argmin(ori_curve_dydx, axis=1)
        unique_indices, counts = np.unique(min_indices, return_counts=True)
        significant_indices = unique_indices[counts > (len(ori_curve_dydx) / 3)]
        max_significant_index = int(np.max(significant_indices)) if len(significant_indices) > 0 else None

    t_full = well.time_npr
    X_2d_trunc, t_trunc = X_2d_bs_active, t_full[well.idx_settled:well.idx_end]
    if max_significant_index is not None and (max_significant_index + 1) < (len(X_time) - 100):
        X_2d_trunc = X_2d_bs_active[:, max_significant_index + 1:]
        t_trunc = t_trunc[max_significant_index + 1:]
    t_trunc = t_trunc - t_trunc[0]
    X_2d_trunc = X_2d_trunc - X_2d_trunc[:, 0:1]

    sg_curves, opt_w = cp01.sg_p4_denoise_curves(X_2d_trunc)
    norm_curves = cp01.normalize_curves_minmax(sg_curves)

    return dict(
        exp_name=exp_name, well_id=WELL_ID, row_idx=row_idx,
        t_full=t_full, raw=raw_full[global_r, global_c, :], lin=lin_full[global_r, global_c, :],
        t_trunc=t_trunc, trunc=X_2d_trunc[row_idx],
        denoised=sg_curves[row_idx], normalised=norm_curves[row_idx],
    )

In [ ]:
per_chip_results = {}
for name in chip_names:
    t0 = time.perf_counter()
    per_chip_results[name] = run_pixel_pipeline(name, TARGET_LABEL)
    r = per_chip_results[name]
    print(f"  {name}: well={r['well_id']} row={r['row_idx']}  ({time.perf_counter() - t0:.1f}s)")

## 5. Temporal calibration -- PC-TTP `anchor="min"` alignment

Reuses `04_cross_dataset_training.py`'s own PC-TTP functions unchanged: each chip's
PC-well Ct (`pc_ttp_per_chip`, disk-cached the same way `04`'s CLI caches it) is compared
to the earliest chip's Ct (`anchor = min(...)`), each curve is front-truncated by the
difference (`shift_to_pc_ttp_anchor`), then all curves are back-truncated to the shortest
resulting duration (`truncate_to_common_duration`). Since the truncated window's own
min/max can differ from the pre-truncation curve, `04` re-normalizes to [0,1] afterwards
for any `_norm` curve type (`combine_group_pc_aligned`, `04_cross_dataset_training.py:328-331`)
-- reproduced here the same way.

In [ ]:
PC_TTP_CURVE_TYPE = "ori_curve_sg_p4_norm"
pc_ttp_cache_dir = Path(config.FINAL_EXP_FOLDER) / "cross_dataset_cv" / LOFO_GROUP / "_cache_pc_ttp"
pc_ttp_cache_dir.mkdir(parents=True, exist_ok=True)
exp_paths = [Path(config.FINAL_EXP_FOLDER) / name for name in chip_names]

pc_ttp = cdt.pc_ttp_per_chip(exp_paths, PC_TTP_CURVE_TYPE, pc_ttp_cache_dir)
anchor = min(pc_ttp.values())
print(f"PC TTP per chip: { {k: round(v, 2) for k, v in pc_ttp.items()} }")
print(f"anchor (min) = {anchor:.2f}")

shifted = {}
for name, r in per_chip_results.items():
    t2, y2, shift = cdt.shift_to_pc_ttp_anchor(
        r['t_trunc'], r['normalised'][np.newaxis, :], pc_ttp.get(name), anchor)
    shifted[name] = (t2, y2[0])
    print(f"  {name}: shift={shift:.2f}")

common_duration = min(t2[-1] - t2[0] for t2, _ in shifted.values())
calibrated = {}
for name, (t2, y2) in shifted.items():
    t3, y3 = cdt.truncate_to_common_duration(t2, y2[np.newaxis, :], common_duration)
    if PC_TTP_CURVE_TYPE.endswith("_norm"):
        y3 = cdt.normalize_curves_minmax(y3)  # matches 04's own re-normalize-after-truncation step
    calibrated[name] = (t3, y3[0])
print(f"common_duration = {common_duration:.2f}s")

## Multi-chip pipeline figure (report)

2x3 grid, one panel per stage, all chips overlaid per panel in a consistent colour.
Saved as a high-res PNG (300 dpi) to `notebooks/figures/`.

In [ ]:
%matplotlib inline
STAGE_SPECS = [
    ('Raw Curves',                     't_full',  'raw'),
    ('1. Linearised Curves',           't_full',  'lin'),
    ('2. Truncated Curves',            't_trunc', 'trunc'),
    ('3. Denoised Curves',             't_trunc', 'denoised'),
    ('4. Normalised Curves',           't_trunc', 'normalised'),
    ('5. Temporal Calibrated Curves',  None,      'calibrated'),
]

_PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
chip_colors = {name: _PALETTE[i % len(_PALETTE)] for i, name in enumerate(chip_names)}

def short_chip_name(name):
    return name.split('_U_', 1)[1]

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), dpi=300)
axes = axes.flatten()
fig.subplots_adjust(wspace=0.3, hspace=0.55, bottom=0.08, top=0.88)

for ax, (caption, t_key, y_key) in zip(axes, STAGE_SPECS):
    for name in chip_names:
        if y_key == 'calibrated':
            t, y = calibrated[name]
        else:
            t, y = per_chip_results[name][t_key], per_chip_results[name][y_key]
        if y_key == 'lin':
            y = y - y[0]  # display-only rebaseline -- per-chip vref offset dwarfs the shape otherwise
        ax.plot(t, y, color=chip_colors[name], lw=1.5, alpha=0.9, label=short_chip_name(name))
    ax.set_xlabel("Time (s)", fontsize=10)
    ax.tick_params(labelsize=9)
    ax.tick_params(axis='x', rotation=45)

fig.canvas.draw()  # finalize subplot positions before reading them for captions

renderer = fig.canvas.get_renderer()
for ax, (caption, _, _) in zip(axes, STAGE_SPECS):
    pos = ax.get_position()
    tb = ax.get_tightbbox(renderer).transformed(fig.transFigure.inverted())
    fig.text((pos.x0 + pos.x1) / 2, tb.y0 - 0.025, caption, ha='center', va='top',
             fontsize=13, fontweight='bold')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=len(chip_names),
          bbox_to_anchor=(0.5, 0.98), fontsize=11, frameon=False)
fig.suptitle(f'Curve Preprocessing Pipeline -- {LOFO_GROUP}  |  target={TARGET_LABEL}',
            fontsize=13, fontweight='bold', y=1.03)

SAVE_DIR = Path('figures')
SAVE_DIR.mkdir(exist_ok=True)
# save_path = SAVE_DIR / f'{LOFO_GROUP}_{TARGET_LABEL}_multi_chip_pipeline.png'
# fig.savefig(save_path, dpi=300, facecolor='white', bbox_inches='tight')
# print(f"Saved high-res figure -> {save_path.resolve()}")
plt.show()